In [3]:
from references import zeng24_config, zeng24_question
from src.retrieval import VRConfig, VectorRetriever, RerankerManager, LLMHybridSummarization
from src.prompts import LLMQueryRewriter, SimplePromptConstructor
from src.llm import OpenAILLM
import os
import json
from src.utils import get_retrieval_info, get_data_chunks, get_data_chunks_by_params, get_llm_output_file
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 初始化设置以及数据库

In [7]:
cfg = zeng24_config.Zeng24fiqa()

retrieval_name, retrieval_store_path = get_retrieval_info(cfg)
print(f"Retrieval store path: {retrieval_store_path}")

retriever_config = VRConfig()

retriever_config.update_4m_dict({
    "data": {
            "retrieval_name": retrieval_name,
            "retrieval_store_path": retrieval_store_path,
            "force_rebuild": False,
            "datastorage_tool": "chroma",
            "data_dir_list": cfg.datastorage.raw_data_dir,
        },
    "retrieval": {
            "method": cfg.retrieval.method,
            "top_k": cfg.retrieval.params.get("k", 15),
            "fetch_k": cfg.retrieval.params.get("fetch_k", 60),
            "score_threshold": cfg.retrieval.params.get("score_threshold", 0.75)
        },
    "embed": {
            "provider": cfg.embedding.provider,
            "model_dir": cfg.embedding.model_dir
        }
})


# 初始化
retriever = VectorRetriever(retriever_config, device='cuda:1')

Retrieval store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Loading existing Chroma DB: ./data/fiqa
Retriever of mmr is ready.
Retriever of chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!


# 初始化LLM对话

In [8]:
# LL_Model = OpenAILLM(
#                     model = "./Models/Qwen2.5-14B-Instruct", 
#                     base_url = "http://localhost:22999/v1", 
#                     api_key = "EMPTY", 
#                     reasoning= cfg.llm.reasoning,
#                     temperature= cfg.llm.temperature,
#                     top_p= cfg.llm.top_p,
#                     max_gen_len= cfg.llm.max_gen_len)

LL_Model = OpenAILLM(
                    model = "./Models/Qwen3-14B", 
                    base_url = "http://localhost:22999/v1", 
                    api_key = "EMPTY", 
                    reasoning= cfg.llm.reasoning,
                    temperature= cfg.llm.temperature,
                    top_p= cfg.llm.top_p,
                    max_gen_len= cfg.llm.max_gen_len)

# 生成或加载问题

In [9]:
# 输入查询
queries = ["What are the causes of Volume?", "What is Profit Margin?"]

In [10]:
qrw = LLMQueryRewriter(LL_Model)

In [11]:
queries_rws = qrw.rewrite(queries, n_variants=5)

In [13]:
queries_rws

{'original_query': ['What are the causes of Volume?',
  'What is Profit Margin?'],
 'rewritten_queries': [['What are the factors contributing to the increase in volume?',
   'What are the primary causes of volume changes in different contexts?',
   'What are the reasons that might lead to a decrease in volume?',
   'What are the underlying reasons for variations in volume levels?',
   'How do external factors influence volume compared to internal factors?'],
  ['What is the definition of profit margin in business finance?',
   'How is profit margin calculated and what factors influence it?',
   'What are the advantages and disadvantages of a high profit margin for a company?',
   "What does the term profit margin mean and how can it be used to assess a company's financial health?",
   'How does profit margin differ from other financial metrics like return on investment?']],
 'all_queries': [['What are the causes of Volume?',
   'What are the factors contributing to the increase in volu

# 检索得到chunk

In [15]:
reranker = RerankerManager(reranker_model=cfg.retrieval.rerank, top_n=cfg.retrieval.params.get("n", 10), device='cuda:1')

[INFO] Reranker BAAI/bge-reranker-large is ready!


In [10]:
# # 调用 retrieve 方法
# contexts, doc_ids = retriever.retrieve(queries_rws["original_query"])

# # 查看结果
# for i, q in enumerate(queries_rws["original_query"]):
#     print(f"\n🔍 Query: {q}")
#     for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
#         print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")

In [11]:
# contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# # 查看结果
# for i, q in enumerate(queries):
#     print(f"\n🔍 Query: {q}")
#     for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
#         print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")

#### 下面测试使用rewriter的格式

In [18]:
# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries_rws["all_queries"])

# 查看结果
for i, q in enumerate(queries_rws["all_queries"]):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: ['What are the causes of Volume?', 'What are the factors contributing to the increase in volume?', 'What are the primary causes of volume changes in different contexts?', 'What are the reasons that might lead to a decrease in volume?', 'What are the underlying reasons for variations in volume levels?', 'How do external factors influence volume compared to internal factors?']
  1. [569627] Volumes are used to predict momentum of movement, not the direction of it. Large trading volumes gen...
  2. [261802] Open, high, low, close, volume. The hint is that volume on new years day is 0.  DC's comment is actu...
  3. [75680] "There are several causes of inflation. One is called cost push — that is, if the price of e.g. oil ...
  4. [446997] As far as I knew a similar law was already on the books, something about the commercial can be no lo...
  5. [79807] The daily Volume is usually compared to the average daily volume over the past 50 days for a stock. ...
  6. [51311] There are m

In [20]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: What are the causes of Volume?
  1. [569627] Volumes are used to predict momentum of movement, not the direction of it. Large trading volumes gen...
  2. [256717] I would say the real story is less about the implications of low vol but rather what has caused it. ...
  3. [122018] Summarized article: The Department of Commerce reported that retail sales increased 1.1% in Septembe...
  4. [112678] I am no expert, but lots of things can cause drops.  Large unsecured revolving debt (credit cards) -...
  5. [512153] Four possible reasons for the difference:...
  6. [10017] You should not look at volume in isolation but look at it together with other indicators and/or the ...
  7. [554255] Assuming constant velocity, inflation is caused by the difference between the growth in the money su...
  8. [513620] There are two reasons to do a reverse split.  Those partial shares will then be turned into cash and...
  9. [252336] Several people have mentioned the obvious: inflation.  But le

In [21]:
contexts

[['Volumes are used to predict momentum of movement, not the direction of it. Large trading volumes generally tend to create a price breakout in either positive or negative direction. Especially in relatively illiquid stocks (like small caps), sudden volume surges can create sharp price fluctuations.',
  "I would say the real story is less about the implications of low vol but rather what has caused it. IMO that would be:  1) lots of money chasing a handful of investments    a) loose monetary policy    b) wealth effects from fantastic returns since the GR    c) consolidation in various sectors (health, energy, tech)  2) rise of low cost index funds (all inflow go into the large swathes of the market so volatility across stocks is dampened)   3) various externalities of expansionary Fed policy    a) resulting low bond yields lead to larger flows into equities     b) low cost of debt feeding buybacks     c) it has been sustained for so long it has had stabilizing effect i.e. predictabili

## Summarization

In [14]:
summar = LLMHybridSummarization(LL_Model, 
                                embed_provider=cfg.embedding.provider,
                                embed_model_dir=cfg.embedding.model_dir,
                                device='cuda:1')

In [15]:
sum_chunks = summar.summarize(contexts, queries)

# 形成结构prompt

In [16]:
p_construct = SimplePromptConstructor()

In [17]:
p_construct.prefix

['context: ', 'question: ', 'answer:']

In [18]:
end_ppt_contexts = p_construct.batch_construct(queries, contexts)
end_ppt_sum = p_construct.batch_construct(queries, sum_chunks)
end_ppt_query = p_construct.batch_construct(queries, [])

In [19]:
end_ppt_contexts

["context: There are several reasons:\n\nVolumes are used to predict momentum of movement, not the direction of it. Large trading volumes generally tend to create a price breakout in either positive or negative direction. Especially in relatively illiquid stocks (like small caps), sudden volume surges can create sharp price fluctuations.\n\nThe factors to consider:\n\nVery difficult to determine. I know the article wasn't meant to be conclusive, so some additional considerations:  * Industries: other industries may be affected differently  * Population growth: if wage increase caused higher population growth, this would affect employment  * Employment demographics: what % of change is attributable to those jobs around the minimum wage level  * Sales (not profit): if the 'extra money to spend' theory holds, one would expect sales to have grown at an abnormal rate for affected industries  * Rate of technological adoption (proxy: CapEx): did businesses start to implement more/new technolo

In [20]:
end_ppt_sum

['context: There are several reasons:\n\nThe sentence does not provide information on the causes of volume. It discusses the effects of volume on price movements, such as large trading volumes creating price breakouts and sudden volume surges in illiquid stocks causing sharp price fluctuations.\n\nThe factors to consider:\n\nIndustries: other industries may be affected differently  \nPopulation growth: if wage increase caused higher population growth, this would affect employment  \nEmployment demographics: what % of change is attributable to those jobs around the minimum wage level  \nSales (not profit): if the \'extra money to spend\' theory holds, one would expect sales to have grown at an abnormal rate for affected industries  \nRate of technological adoption (proxy: CapEx): did businesses start to implement more/new technologies and within what time frame were/will these implemented\n\nThe sentence does not provide information about the causes of Volume.\n\nThe sentence does not m

# 输入LLM进行测试

In [21]:
LL_Model.infer("who are you?")

("Hello! I'm Qwen, a large language model developed by Alibaba Cloud. I can answer questions, create text, and have conversations on a wide range of topics. I'm also multilingual and can assist with various tasks. How can I help you today? 😊",
 'Okay, the user asked, "who are you?" I need to respond clearly. First, I should introduce myself as Qwen, a large language model developed by Alibaba Cloud. I should mention my capabilities, like answering questions, creating text, and having conversations. Also, I need to highlight my multilingual support and the ability to handle various tasks. But I should keep it concise and friendly. Let me make sure I don\'t use any markdown and keep the response natural.')

In [22]:
LL_Model.batch_infer(end_ppt_sum)

(['The provided context does not explicitly state the **causes of "Volume"** (trading volume) itself but discusses **factors that influence changes in volume levels** (e.g., spikes, drops, or variations). Based on the context and general financial principles, the **causes of volume fluctuations** (not the existence of volume) include:  \n\n1. **Company or Economic News**:  \n   - Earnings reports, mergers, acquisitions, or macroeconomic data releases can drive increased trading activity.  \n\n2. **Trends and Market Sentiment**:  \n   - The start or end of a price trend (e.g., a breakout or reversal) often coincides with higher volume.  \n   - Shifts in investor sentiment (e.g., optimism/pessimism) can amplify trading activity.  \n\n3. **Ex-Dividend Events**:  \n   - A sharp drop in price with increased volume may signal a stock going ex-dividend.  \n\n4. **Liquidity and Market Structure**:  \n   - Illiquid stocks may experience sudden volume surges due to limited participation, leading

In [23]:
LL_Model.batch_infer(end_ppt_contexts)

(['The causes of **trading volume** (in the context of financial markets) are multifaceted and influenced by a combination of market dynamics, news events, and investor behavior. Based on the context provided, here are the key factors:\n\n### 1. **News and Information Releases**  \n   - **Company-specific news**: Earnings reports, product launches, management changes, or legal issues can trigger spikes in trading volume as investors react to new information.  \n   - **Economic data**: Releases like GDP reports, employment figures, or interest rate decisions often drive volume as traders adjust positions based on macroeconomic trends.  \n   - **Ex-dividend events**: A sharp drop in volume may occur when a stock goes ex-dividend, as investors sell shares to capture the dividend.  \n\n### 2. **Market Trends and Momentum**  \n   - **Breakouts or breakdowns**: Large volume surges often accompany price breakouts (upward or downward trends) as traders enter or exit positions.  \n   - **Trend 

In [24]:
LL_Model.batch_infer(queries)

(['The term "volume" can refer to different concepts depending on the context. Below are the **causes of volume** in various fields:\n\n---\n\n### **1. Physics (Physical Volume)**\nIn physics, **volume** is the three-dimensional space occupied by an object. The causes of changes in volume depend on the material and conditions:\n- **Thermal Expansion/Contraction**: Heating or cooling a substance (e.g., gases, liquids, solids) can increase or decrease its volume due to changes in molecular motion.\n- **Pressure Changes**: Gases expand or compress under pressure (Boyle’s Law: $ PV = \\text{constant} $).\n- **Phase Changes**: Melting, freezing, or vaporization can alter volume (e.g., ice expands when freezing).\n- **Material Properties**: Denser materials occupy less volume for the same mass.\n\n---\n\n### **2. Economics/Business (Sales Volume)**\nIn business, **volume** often refers to the quantity of goods or services sold. Causes of changes in sales volume include:\n- **Demand and Suppl

In [25]:
answers, reasons = LL_Model.batch_infer(end_ppt_contexts)

In [235]:
# 保存结果
output_dir = cfg.expconfig.output_dir
os.makedirs(output_dir, exist_ok=True)

answers_path = os.path.join(output_dir, get_llm_output_file(cfg))
with open(answers_path, "w", encoding="utf-8") as f_a:
    json.dump(answers, f_a, ensure_ascii=False, indent=2)

reasons_path = answers_path.replace(".json", "_reasoning.json")
with open(reasons_path, "w", encoding="utf-8") as f_r:
    json.dump(reasons, f_r, ensure_ascii=False, indent=2)

In [236]:
reasons_path

'./exp/fiqa/vector-chroma/bge-large-en-v1_5-Qwen2_5-14B-Instruct/mmr-15-BAAI/bge-reranker-large/outputs-Qwen2.5-14B-Instruct-0-4096-4096_reasoning.json'